In [1]:
import pandas as pd
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import os

In [2]:
# Connect to Elasticsearch
client = Elasticsearch("http://localhost:9200")

In [3]:
!curl http://localhost:9200

{
  "name" : "e0d6f366692f",
  "cluster_name" : "docker-cluster",
  "cluster_uuid" : "XZo9CQgATx2-3DIBXLh6uQ",
  "version" : {
    "number" : "8.11.1",
    "build_flavor" : "default",
    "build_type" : "docker",
    "build_hash" : "6f9ff581fbcde658e6f69d6ce03050f060d1fd0c",
    "build_date" : "2023-11-11T10:05:59.421038163Z",
    "build_snapshot" : false,
    "lucene_version" : "9.8.0",
    "minimum_wire_compatibility_version" : "7.17.0",
    "minimum_index_compatibility_version" : "7.0.0"
  },
  "tagline" : "You Know, for Search"
}


In [4]:
# Check if successful
print(client.info())

{'name': 'e0d6f366692f', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'XZo9CQgATx2-3DIBXLh6uQ', 'version': {'number': '8.11.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '6f9ff581fbcde658e6f69d6ce03050f060d1fd0c', 'build_date': '2023-11-11T10:05:59.421038163Z', 'build_snapshot': False, 'lucene_version': '9.8.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [5]:
index_name = "ir2025_documents"

In [6]:
# Delete index if it exists (so you can rerun this block safely)
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

In [7]:
# Define settings: English Analyzer and BM25 Similarity
settings = {
    "settings": {
        "number_of_shards": 1,
        "analysis": {
            "analyzer": {
                "default": {
                    "type": "english"  # Handles stemming, stopwords automatically
                }
            }
        },
        "similarity": {
            "default": {
                "type": "BM25"  # Standard probabilistic model
            }
        }
    },
    "mappings": {
        "properties": {
            "text": { "type": "text" },  # The content of the document
            "doc_id": { "type": "keyword" } # The document ID
        }
    }
}

In [8]:
# Create the index
client.indices.create(index=index_name, body=settings)
print(f"Index '{index_name}' created.")

Index 'ir2025_documents' created.


In [9]:
# Load documents
df_docs = pd.read_csv("data/documents.csv")

In [10]:
# Prepare data for bulk indexing
def generate_actions(df):
    for index, row in df.iterrows():
        yield {
            "_index": index_name,
            "_id": str(row['ID']), # Use the CSV ID as the Elasticsearch ID
            "_source": {
                "doc_id": str(row['ID']),
                "text": row['Text']
            }
        }

In [11]:
# Index the data
print("Indexing documents... this might take a moment.")
bulk(client, generate_actions(df_docs))
print(f"Indexed {len(df_docs)} documents.")

Indexing documents... this might take a moment.
Indexed 18316 documents.


In [12]:
# Load queries
df_queries = pd.read_csv("data/queries.csv")

In [13]:
# Create results folder if it doesn't exist
if not os.path.exists("results"):
    os.makedirs("results")

In [14]:
output_file = "results/phase1_results.txt"
run_name = "ES_BM25_English"

with open(output_file, 'w') as f:
    # Loop through each query
    for index, row in df_queries.iterrows():
        q_id = row['ID']
        q_text = row['Text']
        
        # Search query
        response = client.search(
            index=index_name,
            query={
                "match": {
                    "text": q_text
                }
            },
            size=50
        )
        
        # Write hits to file in TREC format
        rank = 1
        for hit in response['hits']['hits']:
            doc_id = hit['_id']
            score = hit['_score']
            
            f.write(f"{q_id} Q0 {doc_id} {rank} {score} {run_name}\n")
            rank += 1

print(f"Search complete. Results saved to {output_file}")

Search complete. Results saved to results/phase1_results.txt
